# AETHER — Spike: Moshi as a teacher (feasibility discovery)

Это **не** очередной пронумерованный этап, а разведка перед Stage 5. Мы не знаем — и не можем
проверить офлайн — какой именно вызов публичного пакета `moshi` даёт Mimi audio-токены для
произвольного заданного текста: `moshi` устроен как full-duplex conversational loop (кормишь живым
аудио, получаешь живое аудио), а не как TTS-вызов "текст → токены".

Этот notebook грузит реальные веса Moshi, **не гадает** конкретный API, а печатает его реальную
публичную поверхность (методы, сигнатуры, докстринги) и пробует несколько правдоподобных вызовов,
независимо друг от друга — каждый со своим success/failure в отчёте. Результат — сырой материал для
следующего шага, не готовое решение.

In [ ]:
REPO_URL = "https://github.com/YOUR_USERNAME/YOUR_REPO.git"  # @param {type:"string"}
BRANCH = "main"  # @param {type:"string"}
HF_REPO = "kyutai/moshiko-pytorch-bf16"  # @param {type:"string"}
TEXT = "The weather in Almaty is rainy and twenty four degrees."  # @param {type:"string"}

if "YOUR_USERNAME" in REPO_URL:
    raise ValueError("Укажи настоящий REPO_URL")


In [ ]:
import os, subprocess, sys
from pathlib import Path

subprocess.run(["nvidia-smi"], check=False)
repo_dir = Path("/content/aether")
if (repo_dir / ".git").exists():
    subprocess.run(["git", "-C", str(repo_dir), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(repo_dir)], check=True)
os.chdir(repo_dir)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{repo_dir}[dev,ml,audio]"], check=True)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--upgrade", "torch", "torchvision", "torchaudio"],
    check=True,
)
print("Commit:")
subprocess.run(["git", "rev-parse", "HEAD"], check=True)


In [ ]:
artifacts = repo_dir / "artifacts" / "spike-moshi-teacher"
artifacts.mkdir(parents=True, exist_ok=True)
env = os.environ.copy()
env["PYTHONPATH"] = str(repo_dir / "src")
command = [
    sys.executable, "-m", "aether.experiments.spike_moshi_teacher",
    "--allow-download",
    "--hf-repo", HF_REPO,
    "--text", TEXT,
    "--output-dir", str(artifacts),
]
run = subprocess.run(command, env=env, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
print(run.stdout)
print("Exit code:", run.returncode)


In [ ]:
import json

report_path = artifacts / "report.json"
report = json.loads(report_path.read_text(encoding="utf-8"))
print("Status:", report.get("status"))
print("Conclusion note:", report.get("conclusion_note"))
for attempt in report.get("attempts", []):
    print("-", attempt["attempt"], "succeeded:" , attempt["succeeded"], attempt.get("error", ""))


## Пришли `report.json` целиком

Особенно `lm_surface`, `lm_gen_surface`, `checkpoint_info_surface` — по именам методов и их
сигнатурам решим, какой конкретно вызов использовать как учителя для Stage 5.

In [ ]:
import shutil
from google.colab import files

archive = shutil.make_archive("/content/aether-spike-moshi-teacher", "zip", root_dir=artifacts)
print(archive)
files.download(archive)
